# 🧪 Complete End-to-End Data Science & Tabular Workflow

This sample notebook demonstrates:
1. Loading tabular datasets into Pandas DataFrames.
2. Intercepting DataFrame outputs into the **Tabular Virtualized Grid**.
3. Running in-memory **DuckDB SQL queries** over live DataFrames.
4. Statistical outlier detection and analytical summary.

In [1]:
# Cell 1: Synthetic Customer Dataset Generation
import pandas as pd
import numpy as np

np.random.seed(42)
n_records = 500

departments = ['Engineering', 'Data Science', 'Marketing', 'Finance', 'Sales', 'Product']
regions = ['North America', 'EMEA', 'APAC', 'LATAM']

df_employees = pd.DataFrame({
    'employee_id': [f'EMP-{1000 + i}' for i in range(n_records)],
    'department': np.random.choice(departments, size=n_records),
    'region': np.random.choice(regions, size=n_records),
    'experience_years': np.random.exponential(scale=4, size=n_records).round(1),
    'salary_usd': np.random.normal(loc=115000, scale=28000, size=n_records).astype(int),
    'performance_rating': np.random.choice([1, 2, 3, 4, 5], p=[0.05, 0.15, 0.50, 0.20, 0.10], size=n_records),
    'remote_ratio': np.random.choice([0.0, 0.5, 1.0], p=[0.2, 0.3, 0.5], size=n_records)
})

print(f'Successfully synthesized {len(df_employees)} employee records.')

In [2]:
# Cell 2: Display DataFrame — Triggers Jörmungandr's Virtual Tabular Grid Inspector
df_employees.head(20)

In [3]:
# Cell 3: In-Memory DuckDB Aggregation (with Pandas groupby fallback)
try:
    import duckdb
    query = """
        SELECT 
            department,
            COUNT(*) AS total_staff,
            ROUND(AVG(salary_usd), 2) AS avg_salary,
            ROUND(AVG(experience_years), 1) AS avg_experience,
            ROUND(AVG(performance_rating), 2) AS avg_rating
        FROM df_employees
        GROUP BY department
        ORDER BY avg_salary DESC;
    """
    df_summary = duckdb.query(query).to_df()
    print('Computed aggregation via in-memory DuckDB engine:')
except ImportError:
    print('DuckDB not found; computing aggregation via native Pandas groupby:')
    df_summary = df_employees.groupby('department').agg(
        total_staff=('employee_id', 'count'),
        avg_salary=('salary_usd', lambda x: round(x.mean(), 2)),
        avg_experience=('experience_years', lambda x: round(x.mean(), 1)),
        avg_rating=('performance_rating', lambda x: round(x.mean(), 2))
    ).reset_index().sort_values(by='avg_salary', ascending=False)

df_summary

In [4]:
# Cell 4: Statistical Outlier Telemetry
q1 = df_employees['salary_usd'].quantile(0.25)
q3 = df_employees['salary_usd'].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

outliers = df_employees[(df_employees['salary_usd'] < lower_fence) | (df_employees['salary_usd'] > upper_fence)]
print(f'Salary IQR: ${iqr:,.2f} | Bounds: [${lower_fence:,.2f}, ${upper_fence:,.2f}]')
print(f'Detected {len(outliers)} statistical outlier records.')

## Programmatic IDE Extension Showcase
The cell below uses the `jormungandr` SDK to programmatically launch the **DataFrame Studio** virtualized grid and the **Database Studio** query console.

In [5]:
# Ensure samples directory is in sys.path
import sys, os
for _p in [os.path.abspath('..'), os.path.abspath('.'), os.path.abspath('../..')]:
    if _p not in sys.path:
        sys.path.insert(0, _p)
try:
    import jormungandr as jm
except ImportError:
    import json
    class _FallbackJm:
        @staticmethod
        def _trigger(action, **kwargs):
            payload = {'action': action, **kwargs}
            print('__JG_IDE_ACTION__' + json.dumps(payload) + '__JG_IDE_ACTION_END__', flush=True)
            print(f"[Jormungandr Studio Trigger] {kwargs.get('title', action)} -> Studio activated.")
        @classmethod
        def show_dataframe(cls, *a, **kw): cls._trigger('show_dataframe', **kw)
        @classmethod
        def show_lakehouse(cls, *a, **kw): cls._trigger('show_lakehouse', **kw)
        @classmethod
        def show_pipeline_lineage(cls, *a, **kw): cls._trigger('show_pipeline_lineage', **kw)
        @classmethod
        def show_database_studio(cls, *a, **kw): cls._trigger('show_database_studio', **kw)
        @classmethod
        def show_model_inspector(cls, *a, **kw): cls._trigger('show_model_inspector', **kw)
        @classmethod
        def show_gpu_monitor(cls, *a, **kw): cls._trigger('show_gpu_monitor', **kw)
        @classmethod
        def show_prompt_studio(cls, *a, **kw): cls._trigger('show_prompt_studio', **kw)
        @classmethod
        def show_r_console(cls, *a, **kw): cls._trigger('show_r_console', **kw)
        @classmethod
        def show_plots(cls, *a, **kw): cls._trigger('show_plots', **kw)
        @classmethod
        def show_dag(cls, *a, **kw): cls._trigger('show_dag', **kw)
    jm = _FallbackJm()
    sys.modules['jormungandr'] = jm
    sys.modules['jm'] = jm

# 1. Launch Jormungandr DataFrame Studio with sorting, filtering, and stats
jm.show_dataframe(df_employees, title='Employee Directory - DataFrame Studio')

# 2. Launch Jormungandr Database Studio SQL console
jm.show_database_studio(connection='DuckDB (In-Memory Lakehouse)', query='SELECT department, COUNT(*), AVG(salary_usd) FROM df_employees GROUP BY 1')

print('[Jormungandr] Successfully launched DataFrame Studio and Database Studio!')